# Part B – Hypothesis Testing on Health Records

Significance level used for all tests: **α = 0.05**

Decision rule: if p-value < 0.05 (or |test statistic| > critical value) → **Reject H₀**, otherwise **Accept H₀**.

In [3]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.weightstats import ztest

df = pd.read_csv("health_records.csv")
df.head()

,record_id,age_group,age,weight,gender,region,smoking_status,exercise_frequency,bmi,blood_pressure,diabetes,hypertension,cholesterol_level,glucose_level,visit_date
0,8b8148f6-b38a-488c-a65e-d389b74d0fb1,26-35,32,92,Male,East,Non-Smoker,Daily,26.8,108.2,False,False,204.1,83.5,2024-02-12
1,d8f56413-5be6-428e-98c2-67976142ea7d,60+,71,83,Male,East,Former Smoker,Daily,26.6,126.9,False,False,231.7,99.8,2023-07-09
2,4a15544d-c5e7-4e8a-ba57-8a8ea9488d99,60+,62,72,Female,North,Non-Smoker,Weekly,25.0,115.1,False,False,166.2,89.1,2023-04-04
3,88bd6407-2bcf-4e01-a28d-efe39bf00273,26-35,28,60,Female,North,Non-Smoker,Weekly,25.4,112.8,False,False,220.3,71.2,2023-05-27
4,9132b63e-f162-47e4-a9c3-49e03602f8ac,46-60,49,79,Male,West,Non-Smoker,Never,26.1,120.3,False,False,249.9,119.7,2023-05-16


In [4]:
df.describe()

,age,weight,bmi,blood_pressure,cholesterol_level,glucose_level
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,51.400000,75.497000,26.043900,130.408500,203.050600,98.500100
std,19.193134,15.630749,4.256278,14.047348,31.430234,22.930309
min,18.000000,40.000000,15.800000,87.400000,110.000000,60.000000
25%,35.750000,64.000000,23.100000,120.075000,180.500000,83.575000
50%,51.000000,75.000000,25.900000,130.300000,203.300000,96.050000
75%,67.000000,86.000000,28.800000,140.700000,224.500000,111.125000
max,85.000000,135.000000,42.300000,169.600000,308.300000,217.200000


## 1. Hypotheses

| # | H₀ (Null) | H₁ (Alternative) | Test |
|---|---|---|---|
| 1 | Mean BMI is the same for diabetic and non-diabetic people | Mean BMI is different | z-test |
| 2 | Mean blood pressure is the same for smokers and non-smokers (age 18-25) | Mean blood pressure is different | t-test |
| 3 | Smoking has no effect on diabetes | Smoking affects diabetes | Chi-square |
| 4 | Smoking has no effect on hypertension | Smoking affects hypertension | Chi-square |
| 5 | Hypertension rate is the same in all age groups | At least one age group differs | ANOVA |
| 6 | Diabetes rate is the same in all age groups | At least one age group differs | ANOVA |
| 7 | There is no correlation between age and BMI | There is a correlation | Pearson correlation |

## 2. Confidence Intervals (95%)

CI = mean ± t × (std / √n)

In [5]:
cols = ["age", "weight", "bmi", "blood_pressure", "cholesterol_level", "glucose_level"]
n = len(df)
t_crit = stats.t.ppf(0.975, df=n - 1)

rows = []
for col in cols:
    mean = df[col].mean()
    se = df[col].std() / np.sqrt(n)
    rows.append([col, round(mean, 2), round(mean - t_crit * se, 2), round(mean + t_crit * se, 2)])

pd.DataFrame(rows, columns=["variable", "mean", "lower_95", "upper_95"])

,variable,mean,lower_95,upper_95
0,age,51.40,50.21,52.59
1,weight,75.50,74.53,76.47
2,bmi,26.04,25.78,26.31
3,blood_pressure,130.41,129.54,131.28
4,cholesterol_level,203.05,201.10,205.00
5,glucose_level,98.50,97.08,99.92


We are 95% confident that the true population mean of each variable lies between the lower and upper value.
For example, the average age of the population lies between about 50.2 and 52.6 years.

## 3. Critical Values

For a two-tailed test at α = 0.05:

In [6]:
alpha = 0.05
print("z critical value         :", round(stats.norm.ppf(1 - alpha / 2), 3))
print("t critical value (df=30) :", round(stats.t.ppf(1 - alpha / 2, df=30), 3))
print("Chi-square critical (df=2):", round(stats.chi2.ppf(1 - alpha, df=2), 3))

z critical value         : 1.96
t critical value (df=30) : 2.042
Chi-square critical (df=2): 5.991


Each test below prints its own critical value and p-value, and the decision is made by comparing them.

## 4. z-test / t-test

Rule: if both groups have n ≥ 30 → **z-test**, otherwise → **t-test**.

In [7]:
def compare_means(a, b):
    if len(a) >= 30 and len(b) >= 30:
        stat, p = ztest(a, b)
        crit = stats.norm.ppf(0.975)
        test = "z-test"
    else:
        stat, p = stats.ttest_ind(a, b)
        crit = stats.t.ppf(0.975, df=len(a) + len(b) - 2)
        test = "t-test"

    print(f"{test}: n1={len(a)}, n2={len(b)}")
    print(f"mean1={a.mean():.2f}, mean2={b.mean():.2f}")
    print(f"statistic={stat:.3f}, critical value=±{crit:.3f}, p-value={p:.4g}")
    print("Result:", "Reject H0" if p < 0.05 else "Accept H0")

### Hypothesis 1: BMI of diabetic vs non-diabetic people

In [8]:
diabetic = df[df["diabetes"] == True]["bmi"]
non_diabetic = df[df["diabetes"] == False]["bmi"]
compare_means(diabetic, non_diabetic)

z-test: n1=91, n2=909
mean1=28.80, mean2=25.77
statistic=6.622, critical value=±1.960, p-value=3.551e-11
Result: Reject H0


**Result: Reject H₀.** Both groups are large, so a z-test is used. |z| is greater than 1.96 and p < 0.05,
so diabetic people have a significantly higher mean BMI than non-diabetic people.

### Hypothesis 2: Blood pressure of smokers vs non-smokers (age group 18-25)

In [9]:
young = df[df["age_group"] == "18-25"]
smokers = young[young["smoking_status"] == "Smoker"]["blood_pressure"]
non_smokers = young[young["smoking_status"] == "Non-Smoker"]["blood_pressure"]
compare_means(smokers, non_smokers)

t-test: n1=24, n2=64
mean1=120.19, mean2=114.90
statistic=1.939, critical value=±1.988, p-value=0.05581
Result: Accept H0


**Result: Accept H₀.** The smoker group has only 24 people, so a t-test is used. Smokers have a slightly higher
mean blood pressure, but |t| is below the critical value and p > 0.05, so the difference is not significant.

## 5. Chi-square Test (Smoking vs Disease)

In [10]:
def chi_square(col1, col2):
    table = pd.crosstab(df[col1], df[col2])
    print(table, "\n")
    chi2, p, dof, expected = stats.chi2_contingency(table)
    crit = stats.chi2.ppf(0.95, dof)
    print(f"chi2={chi2:.3f}, dof={dof}, critical value={crit:.3f}, p-value={p:.4g}")
    print("Result:", "Reject H0" if p < 0.05 else "Accept H0")

### Hypothesis 3: Smoking vs Diabetes

In [11]:
chi_square("smoking_status", "diabetes")

diabetes        False  True 
smoking_status              
Former Smoker     196     15
Non-Smoker        533     56
Smoker            180     20 

chi2=1.325, dof=2, critical value=5.991, p-value=0.5155
Result: Accept H0


**Result: Accept H₀.** χ² is less than the critical value and p > 0.05.
Smoking status has no significant effect on diabetes in this data.

### Hypothesis 4: Smoking vs Hypertension

In [12]:
chi_square("smoking_status", "hypertension")

hypertension    False  True 
smoking_status              
Former Smoker     144     67
Non-Smoker        436    153
Smoker            122     78 

chi2=12.594, dof=2, critical value=5.991, p-value=0.001842
Result: Reject H0


**Result: Reject H₀.** χ² is greater than the critical value and p < 0.05.
Smoking status is associated with hypertension (smokers have a higher hypertension rate).

## 6. ANOVA (Age Group vs Disease Rate)

Diseases are coded as 1 (yes) / 0 (no), so the mean of each age group is its disease rate.

In [13]:
def anova(disease):
    groups = [g[disease].astype(int) for _, g in df.groupby("age_group")]
    print(df.groupby("age_group")[disease].mean().round(3), "\n")
    f, p = stats.f_oneway(*groups)
    k, n = len(groups), len(df)
    crit = stats.f.ppf(0.95, k - 1, n - k)
    print(f"F={f:.3f}, critical value={crit:.3f}, p-value={p:.4g}")
    print("Result:", "Reject H0" if p < 0.05 else "Accept H0")

### Hypothesis 5: Hypertension rate across age groups

In [14]:
anova("hypertension")

age_group
18-25    0.117
26-35    0.075
36-45    0.121
46-60    0.248
60+      0.561
Name: hypertension, dtype: float64 

F=60.902, critical value=2.381, p-value=4.755e-46
Result: Reject H0


**Result: Reject H₀.** F is much larger than the critical value and p < 0.05.
Hypertension rate differs significantly between age groups and increases with age (highest in 60+).

### Hypothesis 6: Diabetes rate across age groups

In [15]:
anova("diabetes")

age_group
18-25    0.058
26-35    0.048
36-45    0.040
46-60    0.108
60+      0.132
Name: diabetes, dtype: float64 

F=4.601, critical value=2.381, p-value=0.001101
Result: Reject H0


**Result: Reject H₀.** F is greater than the critical value and p < 0.05.
Diabetes rate also differs between age groups, with the older groups (46-60 and 60+) having higher rates.

## 7. Covariance and Correlation

In [16]:
cols = ["age", "weight", "bmi", "blood_pressure", "cholesterol_level", "glucose_level"]
df[cols].cov().round(2)

,age,weight,bmi,blood_pressure,cholesterol_level,glucose_level
age,368.38,40.37,12.56,173.70,230.32,129.77
weight,40.37,244.32,55.85,57.56,104.42,108.80
bmi,12.56,55.85,18.12,19.81,30.01,31.91
blood_pressure,173.70,57.56,19.81,197.33,142.59,84.05
cholesterol_level,230.32,104.42,30.01,142.59,987.86,132.59
glucose_level,129.77,108.80,31.91,84.05,132.59,525.80


In [17]:
df[cols].corr().round(2)

,age,weight,bmi,blood_pressure,cholesterol_level,glucose_level
age,1.00,0.13,0.15,0.64,0.38,0.29
weight,0.13,1.00,0.84,0.26,0.21,0.30
bmi,0.15,0.84,1.00,0.33,0.22,0.33
blood_pressure,0.64,0.26,0.33,1.00,0.32,0.26
cholesterol_level,0.38,0.21,0.22,0.32,1.00,0.18
glucose_level,0.29,0.30,0.33,0.26,0.18,1.00


### Hypothesis 7: Age vs BMI

In [18]:
cov = np.cov(df["age"], df["bmi"])[0, 1]
r, p = stats.pearsonr(df["age"], df["bmi"])
print(f"Covariance={cov:.3f}, Correlation r={r:.3f}, p-value={p:.4g}")
print("Result:", "Reject H0" if p < 0.05 else "Accept H0")

Covariance=12.557, Correlation r=0.154, p-value=1.041e-06
Result: Reject H0


**Result: Reject H₀.** The correlation is statistically significant (p < 0.05), but r ≈ 0.15 means it is a
**weak positive** relationship: BMI increases only slightly with age.

From the correlation table, the strongest relationships are **weight vs BMI** (strong positive) and
**age vs blood pressure** (moderate positive).

## 8. Summary

| # | Hypothesis | Test | Result |
|---|---|---|---|
| 1 | BMI differs between diabetic and non-diabetic | z-test | Reject H₀ |
| 2 | Blood pressure differs between smokers and non-smokers (18-25) | t-test | Accept H₀ |
| 3 | Smoking affects diabetes | Chi-square | Accept H₀ |
| 4 | Smoking affects hypertension | Chi-square | Reject H₀ |
| 5 | Hypertension rate differs across age groups | ANOVA | Reject H₀ |
| 6 | Diabetes rate differs across age groups | ANOVA | Reject H₀ |
| 7 | Age and BMI are correlated | Pearson correlation | Reject H₀ (weak positive) |